# 01 – Viewer Audit

**Objective:** Enumerate confirmed and provisional data sources from the Gdynia thermal viewer
(`termalne.obliview.com`) and produce a source inventory table saved to
`outputs/tables/source_inventory.csv`.

In [ ]:
from __future__ import annotations

import pandas as pd
import requests
import yaml
from pathlib import Path

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent  # adjust if running from a different location
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "tables" / "source_inventory.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open() as fh:
    cfg = yaml.safe_load(fh)

print("Config loaded from:", CONFIG_PATH)

## 1. Parse source definitions from config

In [ ]:
sources = cfg.get("sources", {})
rows = []

for key, val in sources.items():
    if isinstance(val, list):
        for item in val:
            rows.append({
                "group": key,
                "name": item.get("name", key),
                "url": item.get("url", ""),
                "type": item.get("type", ""),
                "confirmed": item.get("confirmed", False),
                "local_path": item.get("local_path", ""),
            })
    elif isinstance(val, dict):
        rows.append({
            "group": key,
            "name": key,
            "url": val.get("url", val.get("base_url", "")),
            "type": val.get("type", ""),
            "confirmed": val.get("confirmed", False),
            "local_path": val.get("local_path", ""),
        })

inventory = pd.DataFrame(rows)
inventory

## 2. Probe confirmed URLs (HEAD request)

In [ ]:
TIMEOUT = 10

def probe_url(url: str) -> int | None:
    """Return HTTP status code or None on error."""
    if not url:
        return None
    try:
        r = requests.head(url, timeout=TIMEOUT, allow_redirects=True)
        return r.status_code
    except requests.RequestException:
        return None

# Only probe confirmed sources to avoid unnecessary traffic
mask = inventory["confirmed"] == True  # noqa: E712
inventory.loc[mask, "http_status"] = inventory.loc[mask, "url"].apply(probe_url)
inventory["http_status"] = inventory["http_status"].astype("Int64")
inventory

## 3. Save inventory

In [ ]:
inventory.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(inventory)} rows to {OUTPUT_PATH}")